# Using NABOPB for a PDE

How to use `NABOPB_for_PDE`, the entry point behind `pde_applications/`, on one example: the
1-d Poisson equation with a parameterised right-hand side. The result approximates the
**solution operator**, the map from right-hand side and point $x$ to $u(x)$.

Install and the plain `NABOPB` call are covered in `tutorial_usage.ipynb`.

## 0. Before you start

Install as in `tutorial_usage.ipynb`, step 0, and select `.venv` as the kernel.

`NABOPB_for_PDE` only works with the `_OMP+`, `_COSAMP+` and `_SRLasso+` methods, which run
through KeOps. KeOps compiles C++ kernels on first use (a minute or two, once, needs a C++
compiler) and prints warnings about CUDA and OpenMP on import; these are expected.

In [1]:
import sys, time, warnings
from pathlib import Path
import numpy as np
from collections import Counter

REPO = Path.cwd() if (Path.cwd() / "NABOPB_for_PDE.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SyntaxWarning)      # one upstream file has LaTeX in plain strings
    from NABOPB_for_PDE import NABOPB_for_PDE

print(f"numpy {np.__version__}   repo {REPO.name}/   ready")

[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
[KeOps] Warning : OpenMP library not found, it must be downloaded through Homebrew for apple Silicon chips
[KeOps] Warning : OpenMP support is not available. Disabling OpenMP.


numpy 2.2.6   repo nabopb/   ready


## 1. The problem

$$-u''(x) = f(x) \quad \text{on } [0, 1], \qquad u(0) = u(1) = 0, \qquad
f(x) = a_0 + \sum_{k \neq 0} a_k \cos(2\pi k x).$$

The nine coefficients $a_{-4}, \dots, a_4$ are the parameters. The target is the map
$(x, a_{-4}, \dots, a_4) \mapsto u(x)$, a function of **ten variables**.

The solution is known in closed form; it stands in for the solver until step 6.

In [2]:
K = np.arange(-4, 5)                                   # the nine modes of the right-hand side

def solve_poisson(x, a):
    """u(x) for right-hand side coefficients a.  x: (n,) in [0, 1];  a: (n, 9), one row per x."""
    k = K.astype(float)
    with np.errstate(divide="ignore", invalid="ignore"):              # the k = 0 term is handled separately
        terms = a / (4 * np.pi**2 * k**2) * (np.cos(2 * np.pi * np.outer(x, k)) - 1)
    return a[:, 4] / 2 * x * (1 - x) + np.nansum(terms, axis=1)

# sanity check: f = 1 (only a_0 = 1) gives u = x(1 - x)/2, so u(0) = u(1) = 0 and u(1/2) = 1/8
a_test = np.zeros((3, 9)); a_test[:, 4] = 1.0
for xv, u in zip([0.0, 0.5, 1.0], solve_poisson(np.array([0.0, 0.5, 1.0]), a_test)):
    print(f"   u({xv}) = {u:.4f}")

   u(0.0) = 0.0000
   u(0.5) = 0.1250
   u(1.0) = 0.0000


## 2. Provide the solver in the shape `NABOPB_for_PDE` expects

`handle(x_pde, params, blocksize)` with

| argument | shape | meaning |
|---|---|---|
| `x_pde` | `(n, pde_dims)` | points in the PDE variables, here $x$ |
| `params` | `(m, d - pde_dims)` | one row per right-hand side |
| `blocksize` | | `n = m * blocksize`; rows `i*blocksize` to `(i+1)*blocksize - 1` of `x_pde` belong to `params[i]` |

Returns `n` values. One solve per parameter row serves its whole block of points.

The Chebyshev basis lives on $[-1, 1]$, so the handle maps $x$ to $[0, 1]$.

In [3]:
def handle(x_pde, params, blocksize=1):
    a = np.repeat(params, blocksize, axis=0)[: x_pde.shape[0]]    # parameter row i for block i
    x = (x_pde[:, 0] + 1) / 2                                      # [-1, 1] -> [0, 1]
    return solve_poisson(x, a)

# what a call looks like: 2 right-hand sides, 3 points each
x_demo = np.array([[-1.0], [0.0], [1.0], [-1.0], [0.0], [1.0]])
a_demo = np.zeros((2, 9)); a_demo[0, 4] = 1.0; a_demo[1, 4] = 2.0
print("x_pde shape", x_demo.shape, " params shape", a_demo.shape, " blocksize 3")
print("returns     ", np.round(handle(x_demo, a_demo, blocksize=3), 4), " (f = 1 gives u(1/2) = 0.125, f = 2 gives 0.25)")

x_pde shape (6, 1)  params shape (2, 9)  blocksize 3
returns      [0.    0.125 0.    0.    0.25  0.   ]  (f = 1 gives u(1/2) = 0.125, f = 2 gives 0.25)


## 3. Call `NABOPB_for_PDE`

```python
result = NABOPB_for_PDE(handle, pde_dims, d, basis_flag, gamma, diminc, sparsity_s)
```

- `pde_dims`: number of PDE variables, here `1`. They come first in the ordering; the rest
  are parameters.
- `basis_flag`: one of the `+` methods. The lattice methods fail inside this function.
- `pde_disc` (optional, default `3`): extra evaluation points per PDE variable for each solve.

Progress lines and the solver's residuals are hidden with `redirect_stdout`.

**Reduced parameters**: `s = 40`, `N = 8`. `pde_applications/demo_pois_1d_f.py` uses
`s = 1000`, `N = 64`.

In [4]:
d, pde_dims = 10, 1
gamma = {"type": "full", "N": 8, "w": 1, "sgn": "non-negative"}
diminc = {"type": "default"}

import contextlib, io

np.random.seed(0)
t0 = time.time()
with contextlib.redirect_stdout(io.StringIO()):           # hides progress lines and solver residuals
    result = NABOPB_for_PDE(handle, pde_dims, d, "Cheby_OMP+", gamma, diminc, sparsity_s=40)
print(f"done in {time.time() - t0:.1f}s")

done in 0.5s


## 4. Read the result

Same dictionary as before. `index` column 0 is the $x$ degree, columns 1 to 9 the degrees in
$a_{-4}, \dots, a_4$. `sample_sizes` counts **PDE solves**, not point evaluations.

In [5]:
index, val = result["index"], np.real(result["val"])
solves = int(np.sum(result["sample_sizes"]))

print(f"{index.shape[0]} coefficients from {solves:,} PDE solves")
print("solves per step:", [int(v) for v in result["sample_sizes"]])

deg_x = index[:, 0]
deg_a = index[:, 1:].sum(axis=1)
print("\ndegree in x            :", dict(sorted(Counter(deg_x.tolist()).items())))
print("total degree in a      :", dict(sorted(Counter(deg_a.tolist()).items())))
print(f"largest |coefficient|  : linear in a {np.abs(val[deg_a == 1]).max():.4f},  "
      f"nonlinear in a {np.abs(val[deg_a > 1]).max():.4f}")

print("\nthe six largest terms   (x degree, which a_k, coefficient)")
for i in np.argsort(-np.abs(val))[:6]:
    which = ", ".join(f"a_{K[j]}" for j in np.flatnonzero(index[i, 1:]))
    print(f"   {index[i, 0]:>2}   {which:<8} {val[i]:>10.6f}")

40 coefficients from 1,402 PDE solves
solves per step: [722, 80, 80, 80, 80, 80, 80, 80, 80, 40]

degree in x            : {0: 9, 2: 12, 4: 8, 6: 5, 8: 6}
total degree in a      : {1: 32, 2: 4, 3: 4}
largest |coefficient|  : linear in a 0.0440,  nonlinear in a 0.0005

the six largest terms   (x degree, which a_k, coefficient)
    0   a_0        0.044047
    2   a_0       -0.031488
    0   a_1       -0.012496
    0   a_-1      -0.012392
    2   a_1        0.012222
    2   a_-1       0.012121


Three things to notice:

- The $x$ direction cost 2 of the 722 first-step solves; each parameter direction cost 80. One
  solve gives $u$ at every $x$. Total about 1,400 solves, against about 13,000 evaluations for
  the plain `NABOPB` call on this problem.
- 32 of 40 terms are linear in the parameters; the rest are about 100 times smaller. Poisson
  is linear in $f$.
- All $x$ degrees are even: the solution is symmetric about $x = 1/2$.

## 5. Use the learned operator

Same evaluation helper as `tutorial_usage.ipynb` step 6. Inputs on $[-1, 1]$ in every variable.

In [6]:
def evaluate_chebyshev(index, val, X):
    """Evaluate the approximation at points X of shape (num_points, d) on [-1, 1]^d."""
    T = np.cos(index[None, :, :] * np.arccos(X)[:, None, :])
    scale = np.sqrt(2.0) ** np.count_nonzero(index, axis=1)
    return np.real(np.prod(T, axis=2) @ (val * scale))

# accuracy over many new (x, a) combinations, drawn the Chebyshev way
np.random.seed(7)
X_new = np.sin(np.pi * (np.random.rand(20000, d) - 0.5))
u_true = handle(X_new[:, :1], X_new[:, 1:], blocksize=1)
u_learned = evaluate_chebyshev(index, result["val"], X_new)
print(f"relative L2 error at 20,000 new (x, a) points: "
      f"{np.linalg.norm(u_learned - u_true) / np.linalg.norm(u_true):.3f}")

# one fixed right-hand side, the solution along x
np.random.seed(11)
a_fixed = np.random.rand(9) * 2 - 1
xs = np.array([0.0, 0.25, 0.5, 0.8, 1.0])
X_line = np.hstack([2 * xs[:, None] - 1, np.tile(a_fixed, (5, 1))])
print(f"\none right-hand side, u along x    {'x':>5}{'learned':>10}{'true':>10}")
for xv, ul, ut in zip(xs, evaluate_chebyshev(index, result["val"], X_line), solve_poisson(xs, np.tile(a_fixed, (5, 1)))):
    print(f"{'':<34}{xv:>5}{ul:>10.5f}{ut:>10.5f}")

relative L2 error at 20,000 new (x, a) points: 0.029

one right-hand side, u along x        x   learned      true
                                    0.0   0.00053   0.00000
                                   0.25  -0.00927  -0.00957
                                    0.5  -0.03737  -0.03571
                                    0.8  -0.00400  -0.00337
                                    1.0   0.00053   0.00000


Error of a few percent from 40 coefficients and about 1,400 solves. Boundary values are not
exactly zero; the boundary condition was never imposed, only learned from samples.

## 6. Plug in a real solver

Same problem with `scipy.integrate.solve_bvp`, as in `pde_applications/demo_pois_1d_s.py`.
One solve per parameter row, evaluated at that row's block. Nothing else changes.

In [7]:
from scipy.integrate import solve_bvp

def rhs(x, a):
    return a[4] + sum(a[j] * np.cos(2 * np.pi * K[j] * x) for j in range(9) if K[j] != 0)

def solve_poisson_numeric(x, a, tol=1e-6):
    """Solve -u'' = f(.; a), u(0) = u(1) = 0 numerically; return u at the points x (in [0, 1])."""
    mesh = np.linspace(0, 1, 50)
    sol = solve_bvp(lambda t, y: np.vstack([y[1], -rhs(t, a)]),
                    lambda ya, yb: np.array([ya[0], yb[0]]), mesh, np.zeros((2, mesh.size)), tol=tol)
    return sol.sol(x)[0]

def handle_numeric(x_pde, params, blocksize=1):
    out = np.empty(x_pde.shape[0])
    for i, a in enumerate(params):                                  # one solve per right-hand side ...
        block = slice(i * blocksize, (i + 1) * blocksize)
        out[block] = solve_poisson_numeric((x_pde[block, 0] + 1) / 2, a)   # ... evaluated at its block
    return out

t0 = time.time()
u_num = handle_numeric(X_line[:, :1], a_fixed[None, :], blocksize=5)
per_solve = time.time() - t0
print(f"one solve, five points: {1000 * per_solve:.0f} ms;  max difference to the closed form: "
      f"{np.max(np.abs(u_num - solve_poisson(xs, np.tile(a_fixed, (5, 1))))):.1e}")
total = solves * per_solve
print(f"the {solves:,} solves of step 3 would take about {total:.0f} seconds with this solver, "
      f"ten times that with one solve per sample point")

one solve, five points: 55 ms;  max difference to the closed form: 4.1e-10
the 1,402 solves of step 3 would take about 77 seconds with this solver, ten times that with one solve per sample point


Plain `NABOPB` would need one solve per sample point, about 13,000 here, ten times more. The
heat and Burgers examples in `pde_applications/` cost about a second per solve; there that is
twenty minutes against three hours. Those scripts parallelise the solver with
`multiprocessing.Pool`.

For your own solver: write it like `handle_numeric`, map the PDE variables from $[-1, 1]$ to
your domain, set `pde_dims`, pick a `+` method, start with a small `sparsity_s`.